# Find Your Nearest Healthcare

Datathon submission · v22 · Data provided by Virtue Foundation · Built on Databricks

Find healthcare facilities in India by PIN code or map location and compare the evidence and details for each requested service. This web app supports both English and Hindi.

## Run the example

Run the next code cell to the project tables in `workspace.datathon_silver`. The example searches for **Dialysis** near PIN code **226003**.

In [0]:
import json

search_sql = r"""
WITH requested AS (
  SELECT DISTINCT EXPLODE(FROM_JSON(:services, 'ARRAY<STRING>')) AS service
),
search_point AS (
  SELECT CAST(:origin_lat AS DOUBLE) AS lat, CAST(:origin_lon AS DOUBLE) AS lon
  WHERE :use_map_point = 1
  UNION ALL
  SELECT lat, lon
  FROM workspace.datathon_silver.gold_india_pin_centers
  WHERE :use_map_point = 0 AND pin = :pin
),
matches AS (
  SELECT e.*
  FROM workspace.datathon_silver.gold_india_service_confidence e
  JOIN requested r ON e.service = r.service
),
matching_campuses AS (
  SELECT campus_id,
    SUM(CASE WHEN confidence = 3 THEN 1 ELSE 0 END) AS high_count,
    SUM(CASE WHEN confidence = 2 THEN 1 ELSE 0 END) AS medium_count
  FROM matches
  GROUP BY campus_id
),
distances AS (
  SELECT f.campus_id, f.facility_name, f.address_line1, f.city, f.state, f.pin_clean,
    f.latitude, f.longitude, f.phone, f.location_flags, p.lat AS origin_latitude, p.lon AS origin_longitude,
    m.high_count, m.medium_count,
    2 * 6371 * ASIN(LEAST(1.0, SQRT(
      POWER(SIN(RADIANS(f.latitude - p.lat) / 2), 2)
      + COS(RADIANS(p.lat)) * COS(RADIANS(f.latitude))
        * POWER(SIN(RADIANS(f.longitude - p.lon) / 2), 2)
    ))) AS distance_km
  FROM workspace.datathon_silver.gold_india_facilities f
  JOIN matching_campuses m ON f.campus_id = m.campus_id
  CROSS JOIN search_point p
  WHERE f.eligible_for_distance AND f.latitude IS NOT NULL AND f.longitude IS NOT NULL
),
unavailable AS (
  SELECT TO_JSON(SORT_ARRAY(COLLECT_SET(r.service))) AS services_json
  FROM requested r
  WHERE NOT EXISTS (
    SELECT 1 FROM matches m JOIN distances d ON m.campus_id = d.campus_id
    WHERE m.service = r.service AND m.confidence >= 2
  )
),
excluded_locations AS (
  SELECT COUNT(*) AS excluded_location_count
  FROM workspace.datathon_silver.gold_india_facilities f
  JOIN matching_campuses m ON f.campus_id = m.campus_id
  WHERE SIZE(f.location_flags) > 0
    AND NOT COALESCE(f.eligible_for_distance AND f.latitude IS NOT NULL AND f.longitude IS NOT NULL, FALSE)
),
scores AS (
  SELECT d.*,
    (high_count + 0.7 * medium_count) / (SELECT COUNT(*) FROM requested) AS confidence_score,
    POWER(0.5, distance_km / :distance_half_life_km) AS distance_score
  FROM distances d
),
weighted_scores AS (
  SELECT *,
    high_count + medium_count > 0 AS recommended,
    60 * confidence_score AS evidence_points,
    40 * distance_score AS distance_points
  FROM scores
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY recommended
      ORDER BY evidence_points + distance_points DESC,
        confidence_score DESC, distance_km, campus_id
    ) AS overall_rank,
    ROW_NUMBER() OVER (
      ORDER BY distance_km, confidence_score DESC, campus_id
    ) AS distance_rank,
    ROW_NUMBER() OVER (
      PARTITION BY recommended
      ORDER BY confidence_score DESC, distance_km, campus_id
    ) AS confidence_rank
  FROM weighted_scores
),
top_results AS (
  SELECT *, 'overall' AS ranking, overall_rank AS result_rank
  FROM ranked WHERE recommended AND overall_rank <= 10
  UNION ALL
  SELECT *, 'nearest' AS ranking, distance_rank AS result_rank
  FROM ranked WHERE distance_rank <= 5
  UNION ALL
  SELECT *, 'confidence' AS ranking, confidence_rank AS result_rank
  FROM ranked WHERE recommended AND confidence_rank <= 5
)
SELECT t.ranking AS ranking, t.result_rank AS result_rank, t.facility_name, t.address_line1, t.city, t.state, t.pin_clean,
       ROUND(t.distance_km, 1) AS straight_line_km, r.service AS service,
       COALESCE(m.confidence, 1) AS confidence, COALESCE(m.evidence_reason, 'missing') AS evidence_reason,
       m.source_field, m.evidence_text, m.source_record_id,
       m.counter_evidence_text, m.counter_source_field, m.counter_record_id,
       t.high_count, t.medium_count, t.confidence_score, t.evidence_points, t.distance_points,
       u.services_json AS unavailable_services_json, CAST(NULL AS STRING) AS status,
       t.latitude AS latitude, t.longitude AS longitude, t.phone AS phone,
       t.origin_latitude AS origin_latitude, t.origin_longitude AS origin_longitude,
       t.location_flags, m.source_urls_json, m.counter_source_urls_json, x.excluded_location_count
FROM top_results t
CROSS JOIN requested r
CROSS JOIN unavailable u
CROSS JOIN excluded_locations x
LEFT JOIN matches m ON t.campus_id = m.campus_id AND r.service = m.service

UNION ALL

SELECT CAST(NULL AS STRING), 0, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
       NULL, NULL, NULL, NULL, NULL, NULL, 0, 0, NULL, NULL, NULL, u.services_json,
       CASE WHEN NOT EXISTS (SELECT 1 FROM search_point) THEN 'unknown_pin' ELSE 'no_matches' END,
       NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, x.excluded_location_count
FROM unavailable u
CROSS JOIN excluded_locations x
WHERE NOT EXISTS (SELECT 1 FROM top_results)
ORDER BY ranking, result_rank, service;
"""

results = spark.sql(
    search_sql,
    args={
        "pin": "226003",
        "use_map_point": 0,
        "origin_lat": 0.0,
        "origin_lon": 0.0,
        "services": json.dumps(["dialysis"]),
        "distance_half_life_km": 25,
    },
)

display(
    results.where("ranking = 'overall'")
    .orderBy("result_rank")
    .select(
        "result_rank", "facility_name", "straight_line_km",
        "evidence_points", "distance_points", "evidence_text",
    )
)

result_rank,facility_name,straight_line_km,evidence_points,distance_points,evidence_text
1,Sewa Hospital & Research Centre,3.3,60.0000000000000,36.491247947509336,Provides dialysis services
2,"Precision Urology Hospital, Lucknow",6.8,60.0000000000000,33.163895899261476,Provides dialysis treatment as part of kidney care
3,Regency Multi Super Speciality Hospital,6.9,60.0000000000000,33.013463274563094,Dialysis
4,Dr Ram Manohar Lohia Institute of Medical Sciences,9.0,60.0000000000000,31.162164865156647,Dialysis is offered as a specialty clinic
5,Sushma Hospital,9.2,60.0000000000000,30.967494487617092,Dialysis
6,Mirus Critical Care Centre,9.8,60.0000000000000,30.478031409327052,Provides CRRT and hemodialysis 24x7
7,"Ekana Hospital And Research Centre, Gomtinagar Vistaar",10.5,60.0000000000000,29.911014140305365,Provides dialysis treatment
8,Brajraj Hospital & Superspeciality Centre,3.0,42.0000000000000,36.83327965236434,24/7 Dialysis Facility
9,Charak Hospital And Research Centre,4.0,42.0000000000000,35.79505597850626,24/7 Advance Dialysis Unit
10,Jagrani Hospital,4.1,42.0000000000000,35.73807408299707,nephrology


## How facilities are ranked

The 100 overall score combines up to **60 points** for service evidence and **40 points** for distance.

- **Service evidence.** Each requested service contributes 1 for high confidence, 0.7 for medium confidence, or 0 for low or missing evidence. We average these values across all requested services and multiply them by 60. Each evidence excerpt includes its original dataset field and record ID. Source links are provided at the record level.
- **Distance.** We use the straight-line distance from the entered PIN reference point or the selected map location to the location of the facility. Distance contributes 40 points at 0 km and halves every 25 km. 

| Ranking | Details |
| --- | --- |
| **Overall** | Shows up to 10 facilities by combined score. |
| **Nearest** | Shows up to 5 facilities by distance and may include lower-confidence records. |
| **Confidence** | Shows up to 5 facilities by evidence score, using distance to break ties. |

Overall and Confidence require medium- or high-confidence evidence for at least one requested service, if there are more than one. When a facility only partially matches a search, check the evidence for each service and contact the facility to confirm current availability.

## Appendix A — Data pipeline

The pipeline reads the Virtue Foundation facilities table and the India Post PIN directory. A fresh setup requires read access to those tables and write access to `workspace.datathon_silver`.

1. Run `clean_facilities.py` with `output_schema` set to `workspace.datathon_silver`.
2. Run the PIN-center SQL statement to create `gold_india_pin_centers`.
3. Run `setup_facility_contacts.sql` and `setup_service_confidence.sql` from v22 to refresh the facility and service evidence tables.

The setup scripts replace the corresponding project tables. The search demo above reads the existing tables.

### datathon_silver/clean_facilities.py

~~~~python
# Databricks notebook source
"""Clean the Virtue Foundation snapshot or the corresponding Databricks tables."""

import argparse
from collections import Counter, defaultdict
import csv
import gzip
import hashlib
import json
import math
from pathlib import Path
import re
from statistics import median
from urllib.parse import urlsplit
import uuid

CATALOG = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
FIELDS = ("description", "procedure", "capability", "equipment", "specialties")
DIALYSIS = re.compile(r"\b(?:dialysis|hemodialysis|haemodialysis)\b", re.I)
CITY_ALIASES = {"bangalore": "Bengaluru", "bengaluru": "Bengaluru", "lucknow": "Lucknow"}
STATE_ALIASES = {"up": "Uttar Pradesh", "u.p.": "Uttar Pradesh", "uttar pradesh": "Uttar Pradesh",
                 "ghaziabad, uttar pradesh": "Uttar Pradesh", "orissa": "Odisha", "odisha": "Odisha",
                 "uttarakhand": "Uttarakhand", "uttaranchal": "Uttarakhand"}
ARAVIND_CONFLICT = "b8a5401f-42f1-422a-8cd9-686a15b4cb76"
CHARAK = "a45cac41-6d27-41ab-878b-7dfb89ec8f33"
CHARAK_ALIAS = "2b5d4240-8105-4e60-adb7-01f8cee3a622"
REVIEW_JSON = '''
[
  {
    "source_id": "8b1f6969-a3ce-492a-b855-2c04199f2e5c",
    "status": "supported",
    "location_conflict": false,
    "url": "https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2024-1427_5th%20Edition_Scope%20Correction.pdf",
    "quote": "Nephrology including Dialysis",
    "note": "NABH scope names Fatima Hospital, 35-C Mahanagar, Lucknow 226006, and covers dialysis. Listed validity runs to 2028-09-04. The raw coordinates are about 281 km from the PIN reference and are on hold.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "907f5a8b-088d-484d-82d6-b92a9465f160",
    "status": "historical",
    "location_conflict": false,
    "url": "https://www.dtf.in/wp-content/files/Updated_list_of_empanelled_HCOs__names_of_nodal_officers_nominated_by_the_HCOs_under_CGHS_Lucknow_as_on_04_June_2025_As_on_CGHS_website-16.07.2025.pdf",
    "quote": "Nephrology including dialysis",
    "note": "The CGHS listing names the Vikas Khand site and lists dialysis. Its listed coverage ended 2026-04-30.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "a45cac41-6d27-41ab-878b-7dfb89ec8f33",
    "status": "supported",
    "location_conflict": false,
    "url": "https://www.charakhospital.org/procedures/dialysis",
    "quote": "we provide comprehensive hemodialysis and peritoneal dialysis services with expert nephrology care",
    "note": "Provider confirms both services. The Hardoi Road Dubagga address matches the same campus as the other Charak record.",
    "reviewed_on": "2026-11-21",
    "location_url": "https://www.charakhospital.org/contactus"
  },
  {
    "source_id": "d4066f5a-ad1f-48b9-92cf-6b32431939a8",
    "status": "unverified",
    "location_conflict": false,
    "url": "https://www.drrmlims.ac.in/about/",
    "quote": null,
    "note": "The institute confirms the Vibhuti Khand campus and nephrology. A training-course listing does not independently establish a patient dialysis service at this campus.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "2b5d4240-8105-4e60-adb7-01f8cee3a622",
    "status": "supported",
    "location_conflict": false,
    "url": "https://www.charakhospital.org/procedures/dialysis",
    "quote": "we provide comprehensive hemodialysis and peritoneal dialysis services with expert nephrology care",
    "note": "Same Hardoi Road Dubagga campus as Charak Hospital And Research Centre. The source corroboration is stored once per campus.",
    "reviewed_on": "2026-11-21",
    "location_url": "https://www.charakhospital.org/contactus"
  },
  {
    "source_id": "4e3a5fb5-df76-4d34-81bd-b3448a372f48",
    "status": "conflict",
    "location_conflict": true,
    "url": "https://max-website20-images.s3.ap-south-1.amazonaws.com/Half_Yearly_Report_of_LKO_Apr_25_Sep_25_94c2bac5c0.pdf",
    "quote": null,
    "note": "Operator identifies former Sahara Hospital in Viraj Khand, Gomti Nagar 226010. The dataset row gives Hardoi Road 226003. Hold site and service attribution.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "5fe4b03b-9b0b-4c21-8004-04fdd08ed3c9",
    "status": "conflict",
    "location_conflict": true,
    "url": "https://www.suneyehospital.com/",
    "quote": null,
    "note": "The provider lists Alambagh and Gomti Nagar eye-care sites. The row gives Kakori/Mohan Road and a bare dialysis label. No provider source ties that claim to Sun Eye.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "705cc566-7ec3-40f3-aaec-4b114fc6d2da",
    "status": "supported",
    "location_conflict": true,
    "url": "https://www.balrampurhospital.com/",
    "quote": "A 10-beded dialysis unit",
    "note": "Provider describes a dialysis unit. Corrected PIN 224176 to 226018 using the provider contact page. Facility coordinates still need direct verification.",
    "reviewed_on": "2026-11-21",
    "verified_pin": "226018",
    "location_url": "https://www.balrampurhospital.com/Contact.aspx"
  },
  {
    "source_id": "84f06705-ee19-4ee1-a974-4bc46a797271",
    "status": "supported",
    "location_conflict": false,
    "url": "https://www.sewahospitallko.com/who-we-are.php",
    "quote": "the hospital provisions 100 beds, ICU centre, NICU, PBU, Dialysis, X-Ray",
    "note": "Provider names the service and the Sewa Nagar, Sitapur Road campus.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "a02093bd-9617-4be4-910d-687db2b99a61",
    "status": "conflict",
    "location_conflict": true,
    "url": "https://nishathospital.com/contact-us/",
    "quote": null,
    "note": "The dataset dialysis excerpt explicitly names Isha Hospital Jaunpur. Nishat's official contact page gives Kaisarbagh, Lucknow versus Meerut Road in the row.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "c80ad4c1-cdaa-4254-a07b-7cc0364d7c5d",
    "status": "unverified",
    "location_conflict": false,
    "url": "https://www.krishnamedicalcentre.org/contact-us",
    "quote": null,
    "note": "Provider confirms 1 Rana Pratap Marg. The equipment claim needs a site-specific dialysis source.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "1980afa5-d795-4260-a066-33f36e51aded",
    "status": "unverified",
    "location_conflict": true,
    "url": "https://portal.nabh.co/Documents/AccreditedList/Hospitals/H-2023-1135_5th%20Edition_Surveillance.pdf",
    "quote": null,
    "note": "A dialysis unit appears in the dataset equipment field. The reviewed accreditation source does not establish the service at the dataset's Vishal Khand address.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "bfa6780d-a589-493c-b846-834f232d4efd",
    "status": "conflict",
    "location_conflict": true,
    "url": "https://mau.nic.in/en/public-utility/sharda-narayan-hospital/",
    "quote": null,
    "note": "The district government lists Sharda Narayan in Mau. The dataset labels the city Lucknow and the service excerpt itself mentions Mau. Hold location and service attribution.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "49e46a47-07ec-4234-aabe-07fa94223806",
    "status": "supported",
    "location_conflict": false,
    "url": "https://www.miruscriticalcare.com/mirus-hospital-facilities",
    "quote": "Continuous Renal Replacement Therapy and regular dialysis available 24x7.",
    "note": "The provider confirms dialysis. Its contact page matches the TCV-84-A/86/87 site in Vibhuti Khand.",
    "reviewed_on": "2026-11-21",
    "location_url": "https://www.miruscriticalcare.com/contact-mirus-hospital-lucknow"
  },
  {
    "source_id": "25c6a2b2-af5d-48c7-9ae7-605f04ee3b60",
    "status": "supported",
    "location_conflict": false,
    "url": "https://jims.edu.in/javitri_hospital.aspx",
    "quote": "specialized unit of ICU, NICU, dialysis & Trauma",
    "note": "The operator identifies the dialysis unit at the Telibagh site and separately mentions a fertility unit in Badshah Nagar. The raw address matches Telibagh.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "3a3ce4a6-6a09-4c9d-a8c4-de1ab65754d8",
    "status": "conflict",
    "location_conflict": true,
    "url": "https://www.charakhospital.org/Satellite-opd",
    "quote": null,
    "note": "The operator identifies Charak Polyclinic as a satellite OPD in Raebareli. The dialysis page belongs to the main Lucknow hospital. Keep this site separate and hold the service claim.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "3eea04cf-d1a5-4ac9-8e40-4c7ee0f52858",
    "status": "supported",
    "location_conflict": false,
    "url": "https://precisionuro.com/areas-of-excellance/nephrology-services/renal-failure-management",
    "quote": "At Precision Hospital, ESRD patients are given RRT which includes dialysis and kidney transplantation.",
    "note": "The provider describes dialysis for patients at Precision Hospital. Campus matched by provider identity and Ring Road address.",
    "reviewed_on": "2026-11-21",
    "location_url": "https://precisionuro.com/"
  },
  {
    "source_id": "dbc80639-dbf4-4eb1-bb1e-a4142c4cf41f",
    "status": "unverified",
    "location_conflict": true,
    "url": "https://familyhospitallko.in/",
    "quote": null,
    "note": "The provider confirms the Sitapur Road facility but does not substantiate the equipment excerpt. A directory lists a different PIN, so hold the location pending confirmation.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "413c1fbc-9765-4881-8422-f44601851939",
    "status": "supported",
    "location_conflict": false,
    "url": "https://freseniuskidneycare.asia/center-detail/ind1004/",
    "quote": "Standard hemodialysis (HD)",
    "note": "The service operator lists hemodialysis at Regency Hospital on Tedhi Pulia Ring Road, matching PIN 226022.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "8983f199-31f7-4c8e-b253-79175d2a4f23",
    "status": "unverified",
    "location_conflict": false,
    "url": "https://www.linkedin.com/company/ekana-hospital-and-research-centre",
    "quote": null,
    "note": "The public profile identifies the Sector 4 campus. The dataset service claim needs a verifiable site-specific dialysis source.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "d93e5058-3819-47e3-aaae-49e6d9cfbafb",
    "status": "conflict",
    "location_conflict": true,
    "url": "https://www.practo.com/lucknow/hospital/bnk-hospital-1-csir-colony/services",
    "quote": null,
    "note": "The directory lists dialysis but gives B-1/196 versus B-1/96 in the row. A separate NABH document gives A-3/A-4/A-5 and PIN 226020. Location needs reconciliation.",
    "reviewed_on": "2026-11-21"
  },
  {
    "source_id": "f67f6103-b4a2-451c-ae59-d52cd4696534",
    "status": "supported",
    "location_conflict": false,
    "url": "https://brajrajhospital.com/",
    "quote": "24/7 Dialysis Facility",
    "note": "Provider advertises dialysis and the contact page identifies the Tahseenganj Crossing campus.",
    "reviewed_on": "2026-11-21",
    "location_url": "https://brajrajhospital.com/contact/"
  }
]
'''


def clean(value):
    if value is None:
        return None
    result = " ".join(str(value).split())
    return None if result.casefold() in {"", "null", "none", "nan", "n/a", "na"} else result


def key(value):
    return (clean(value) or "").casefold()


def place(value, aliases):
    return aliases.get(key(value), clean(value))


def pin(value):
    compact = re.sub(r"[\s-]", "", clean(value) or "")
    return compact if re.fullmatch(r"[1-9][0-9]{5}", compact) else None


def number(value):
    try:
        result = float(value)
    except (TypeError, ValueError):
        return None
    return result if math.isfinite(result) else None


def array(value):
    if clean(value) is None:
        return []
    try:
        parsed = value if isinstance(value, (list, tuple)) else json.loads(value)
    except (TypeError, ValueError):
        return []
    return [item for item in parsed if isinstance(item, str) and clean(item)] if isinstance(parsed, (list, tuple)) else []


def website(value):
    value = clean(value) or ""
    try:
        return (urlsplit(value if "://" in value else "//" + value).hostname or "").removeprefix("www.")
    except ValueError:
        return ""


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str, ensure_ascii=False).encode()).hexdigest()


def csv_rows(path):
    csv.field_size_limit(2**27)
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8-sig", newline="") as stream:
        yield from csv.DictReader(stream, strict=True)


def build_pin_index(rows):
    areas, points = defaultdict(set), defaultdict(set)
    for row in rows:
        code = pin(row.get("pincode"))
        state = place(row.get("statename"), STATE_ALIASES)
        district = place(row.get("district"), CITY_ALIASES)
        if code and state and district:
            areas[code].add((key(state), key(district)))
            lat, lon, status = coordinate_status(row)
            if status == "in_bounds":
                points[code].add((lat, lon))
    return areas, points


def distance_km(first_lat, first_lon, second_lat, second_lon):
    dlat = math.radians(second_lat - first_lat)
    dlon = math.radians(second_lon - first_lon)
    h = (math.sin(dlat / 2) ** 2 +
         math.cos(math.radians(first_lat)) * math.cos(math.radians(second_lat)) * math.sin(dlon / 2) ** 2)
    return 6371 * 2 * math.asin(min(1.0, math.sqrt(h)))


def coordinate_status(row):
    lat = number(row.get("latitude"))
    lon = number(row.get("longitude"))
    if lat is None or lon is None:
        if clean(row.get("latitude")) is None or clean(row.get("longitude")) is None:
            return lat, lon, "missing"
        return lat, lon, "invalid"
    if not (6 <= lat <= 38 and 68 <= lon <= 98):
        return lat, lon, "out_of_india_bounds"
    return lat, lon, "in_bounds"


def pin_lookup(code, state, city, index):
    areas = sorted(index.get(code, ()))
    states = sorted({s for s, _ in areas})
    if not code:
        status = "invalid_or_missing"
    elif not states:
        status = "not_found"
    elif key(state) and key(state) not in states:
        status = "state_mismatch"
    elif len(areas) > 1:
        status = "ambiguous"
    elif key(city) == areas[0][1]:
        status = "state_and_city_match"
    else:
        status = "state_match_city_unchecked"
    return status, states, [f"{s} / {d}" for s, d in areas]


def excerpts(row, field):
    if field == "description":
        raw = row.get(field) or ""
        parts = re.split(r"(?<=[.!?])\s+|\r?\n+", raw)
    else:
        parts = array(row.get(field))
    return list(dict.fromkeys(p.strip() for p in parts if DIALYSIS.search(p)))


def clean_data(facility_rows, pin_rows, reviews):
    pin_index, pin_points = build_pin_index(pin_rows)
    groups = defaultdict(dict)
    audit = []
    for i, row in enumerate(facility_rows, 1):
        raw_id = clean(row.get("unique_id"))
        fingerprint = digest(row)
        try:
            raw_id = str(uuid.UUID(raw_id or ""))
        except ValueError:
            audit.append(dict(input_record=i, row_sha256=fingerprint, source_id=None, reason="invalid_facility_id"))
            continue
        if key(row.get("organization_type")) != "facility" or key(row.get("address_countryCode")) != "in":
            audit.append(dict(input_record=i, row_sha256=fingerprint, source_id=raw_id, reason="outside_india_facility_scope"))
            continue
        if fingerprint in groups[raw_id]:
            audit.append(dict(input_record=i, row_sha256=fingerprint, source_id=raw_id, reason="exact_duplicate"))
            continue
        groups[raw_id][fingerprint] = row

    review_by_id = {entry["source_id"]: entry for entry in reviews}
    def chosen(source_id):
        variants = groups[source_id]
        return variants[min(variants)]

    def address_signature(row):
        return (key(row.get("address_line1")), key(row.get("address_line2")),
                pin(row.get("address_zipOrPostcode")), website(row.get("officialWebsite")))

    merged = (CHARAK in groups and CHARAK_ALIAS in groups
              and address_signature(chosen(CHARAK)) == address_signature(chosen(CHARAK_ALIAS))
              and address_signature(chosen(CHARAK)) == ("hardoi road", "dubagga", "226003", "charakhospital.org"))
    duplicate_blocks = defaultdict(list)
    for source_id in groups:
        row = chosen(source_id)
        signature = address_signature(row)
        if all(signature):
            duplicate_blocks[signature].append(source_id)
    duplicate_candidates = {source_id: [other for other in block if other != source_id]
                            for block in duplicate_blocks.values() if len(block) > 1 for source_id in block}
    campus_for = {raw_id: CHARAK if merged and raw_id == CHARAK_ALIAS else raw_id for raw_id in groups}
    campuses = defaultdict(list)
    for raw_id, campus_id in campus_for.items():
        campuses[campus_id].append(raw_id)

    facilities, evidence, review_output = [], [], []
    corroborated = set()
    for campus_id, ids in sorted(campuses.items()):
        ids.sort()
        row = chosen(campus_id)
        name = clean(row.get("name"))
        city = place(row.get("address_city"), CITY_ALIASES)
        state = place(row.get("address_stateOrRegion"), STATE_ALIASES)
        current_review = review_by_id.get(campus_id, {})
        reviewed_pin = current_review.get("verified_pin")
        code = pin(reviewed_pin or row.get("address_zipOrPostcode"))
        pin_status, pin_states, pin_areas = pin_lookup(code, state, city, pin_index)
        lat, lon, status = coordinate_status(row)
        location_notes = []
        pin_reference_km = None
        points = pin_points.get(code)
        if status == "in_bounds" and points and len(pin_areas) == 1 and pin_status != "state_mismatch":
            center_lat = median(x for x, _ in points)
            center_lon = median(y for _, y in points)
            if max(distance_km(center_lat, center_lon, x, y) for x, y in points) <= 30:
                pin_reference_km = round(distance_km(lat, lon, center_lat, center_lon), 3)
                if pin_reference_km > 30:
                    location_notes.append("pin_coordinate_disagreement")
        if pin_status == "state_mismatch":
            location_notes.append("pin_state_mismatch")
        if any(len(groups[raw_id]) > 1 for raw_id in ids):
            location_notes.append("conflicting_source_id_variants")
        if any(review_by_id.get(raw_id, {}).get("location_conflict") for raw_id in ids):
            location_notes.append("reviewed_location_conflict")
        if ARAVIND_CONFLICT in ids:
            location_notes.append("mixed_facility_and_address")
        if status == "in_bounds" and location_notes:
            status = "needs_review"

        possible_duplicates = sorted({campus_for[other] for source_id in ids
                                      for other in duplicate_candidates.get(source_id, [])} - {campus_id})
        facilities.append(dict(
            campus_id=campus_id, name=name, name_key=key(name), city=city, city_key=key(city),
            state=state, state_key=key(state), pin_clean=code, pin_status=pin_status,
            pin_correction_url=current_review.get("location_url") if reviewed_pin else None,
            pin_format_status=("valid" if code else "missing" if clean(row.get("address_zipOrPostcode")) is None else "invalid"),
            pin_states=pin_states, pin_state_district_candidates=pin_areas, pin_reference_km=pin_reference_km,
            latitude=lat if status == "in_bounds" else None,
            longitude=lon if status == "in_bounds" else None, coordinate_status=status,
            location_flags=location_notes, eligible_for_distance=status == "in_bounds",
            official_domain=website(row.get("officialWebsite")), source_ids=ids,
            address_line1=clean(row.get("address_line1")), address_line2=clean(row.get("address_line2")),
            address_line3=clean(row.get("address_line3")), possible_duplicate_campus_ids=possible_duplicates,
            duplicate_review=("confirmed_same_campus" if len(ids) > 1 else
                              "possible_duplicate" if possible_duplicates else "single_source_id"),
            raw_locations=[
                {**{k: str(item[k]) if item.get(k) is not None else None for k in
                    ("name", "address_line1", "address_line2", "address_line3", "address_city", "address_stateOrRegion",
                     "address_zipOrPostcode", "latitude", "longitude", "officialWebsite")},
                 "source_id": source_id, "row_sha256": fp}
                for source_id in ids for fp, item in sorted(groups[source_id].items())
            ],
        ))
        for raw_id in ids:
            review = review_by_id.get(raw_id)
            found = False
            for fingerprint, source_row in sorted(groups[raw_id].items()):
                hits = [(field, excerpt) for field in FIELDS for excerpt in excerpts(source_row, field)]
                for field, excerpt in hits:
                    found = True
                    evidence.append(dict(
                        campus_id=campus_id, source_id=raw_id, service="dialysis", source_field=field,
                        evidence_kind="dataset_excerpt", evidence_text=excerpt, row_sha256=fingerprint,
                        source_url=None, candidate_source_urls=array(source_row.get("source_urls")),
                        evidence_status=("conflict" if raw_id == ARAVIND_CONFLICT or
                                         (review and review["status"] == "conflict") else "unverified"),
                        note=(review["note"] if review else
                              "Mixed providers and addresses in the source record." if raw_id == ARAVIND_CONFLICT else
                              "Keyword match. Verify service and campus together."),
                    ))
            if review and found:
                review_output.append({**review, "campus_id": campus_id, "facility_name": name})
                corroboration_key = (campus_id, review["url"], review["quote"])
                if review.get("quote") and corroboration_key not in corroborated:
                    corroborated.add(corroboration_key)
                    evidence.append(dict(
                        campus_id=campus_id, source_id=raw_id, service="dialysis", source_field="external_page",
                        evidence_kind="external_corroboration", evidence_text=review["quote"], row_sha256=None,
                        source_url=review["url"], candidate_source_urls=[],
                        evidence_status=review["status"], note=review["note"],
                    ))
    for item in evidence:
        item["evidence_id"] = digest(item)
    assert len({r["campus_id"] for r in facilities}) == len(facilities)
    assert len({r["evidence_id"] for r in evidence}) == len(evidence)
    assert all(r["campus_id"] in campuses for r in evidence)
    assert all(r["source_url"] and r["evidence_text"] for r in evidence if r["evidence_status"] == "supported")
    return {"silver_facilities": facilities, "silver_service_evidence": evidence,
            "lucknow_dialysis_review": sorted(review_output, key=lambda r: (r["facility_name"], r["source_id"])),
            "facility_cleaning_audit": audit}


def write_local(results, target):
    target.mkdir(parents=True, exist_ok=True)
    for name, rows in results.items():
        with (target / f"{name}.jsonl").open("w", encoding="utf-8") as output:
            for row in rows:
                print(json.dumps(row, ensure_ascii=False), file=output)


def run_local():
    parser = argparse.ArgumentParser()
    parser.add_argument("facilities", type=Path)
    parser.add_argument("pin_directory", type=Path)
    parser.add_argument("--output", type=Path, default=Path("output"))
    args = parser.parse_args()
    reviews = json.loads(REVIEW_JSON)
    results = clean_data(csv_rows(args.facilities), csv_rows(args.pin_directory), reviews)
    write_local(results, args.output)
    return {name: len(rows) for name, rows in results.items()}

# COMMAND ----------

# Set output_schema to an existing writable catalog.schema before using Run all.
# The project-specific table names in that schema are replaced on a rerun.
if "spark" in globals() and "dbutils" in globals():
    dbutils.widgets.text("output_schema", "")
    output_schema = dbutils.widgets.get("output_schema").strip()
    if not re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*\.[A-Za-z_][A-Za-z_0-9]*", output_schema):
        raise ValueError("Set output_schema to an existing writable catalog.schema.")

    reviews = json.loads(REVIEW_JSON)
    facility_source = spark.table(CATALOG + ".facilities").toLocalIterator()
    pin_source = spark.table(CATALOG + ".india_post_pincode_directory").toLocalIterator()
    results = clean_data((r.asDict(recursive=True) for r in facility_source),
                         (r.asDict(recursive=True) for r in pin_source), reviews)
    for name, rows in results.items():
        if not rows:
            if name == "facility_cleaning_audit":
                empty = spark.createDataFrame([], "input_record INT, row_sha256 STRING, source_id STRING, reason STRING")
                empty.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{output_schema}.{name}")
            else:
                raise ValueError(f"Unexpected empty result for {name}. Check the source tables.")
            print(f"{name}: 0 rows")
            continue
        lines = [json.dumps(row, ensure_ascii=False) for row in rows]
        # Serverless supports Spark Connect DataFrame APIs. Use SQL JSON parsing
        # with an explicit schema inferred from the Cleansing output below.
        from pyspark.sql import functions as F
        from pyspark.sql.types import (ArrayType, BooleanType, DoubleType, IntegerType,
                                       StringType, StructField, StructType)
        def infer(values):
            populated = [v for v in values if v is not None]
            if not populated:
                return StringType()
            example = populated[0]
            if isinstance(example, dict):
                return StructType([StructField(k, infer([v.get(k) for v in populated]))
                                   for k in sorted(set().union(*(v.keys() for v in populated)))])
            if isinstance(example, list):
                return ArrayType(infer([item for v in populated for item in v]))
            if isinstance(example, bool):
                return BooleanType()
            if isinstance(example, int):
                return IntegerType()
            if isinstance(example, float):
                return DoubleType()
            return StringType()
        table = spark.createDataFrame([(line,) for line in lines], "record STRING")
        parsed = table.select(F.from_json("record", infer(rows)).alias("row")).select("row.*")
        parsed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{output_schema}.{name}")
        print(f"{name}: {len(rows)} rows")
elif __name__ == "__main__":
    print(json.dumps(run_local(), indent=2))
~~~~

### setup_pin_centers.sql

~~~~sql
CREATE OR REPLACE TABLE workspace.datathon_silver.gold_india_pin_centers AS
WITH postal_points AS (
  SELECT DISTINCT
    TRIM(CAST(pincode AS STRING)) AS pin,
    TRY_CAST(latitude AS DOUBLE) AS lat,
    TRY_CAST(longitude AS DOUBLE) AS lon
  FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory
  WHERE TRIM(CAST(pincode AS STRING)) RLIKE '^[1-9][0-9]{5}$'
),
valid_points AS (
  SELECT pin, lat, lon
  FROM postal_points
  WHERE lat BETWEEN 6 AND 38 AND lon BETWEEN 68 AND 98
),
pin_reference AS (
  SELECT pin, PERCENTILE(lat, 0.5) AS median_lat,
         PERCENTILE(lon, 0.5) AS median_lon, COUNT(*) AS point_count
  FROM valid_points
  GROUP BY pin
),
near_reference AS (
  SELECT p.pin, p.lat, p.lon, r.point_count
  FROM valid_points p
  JOIN pin_reference r ON p.pin = r.pin
  WHERE 111.2 * SQRT(
    POWER(p.lat - r.median_lat, 2)
    + POWER((p.lon - r.median_lon) * COS(RADIANS(r.median_lat)), 2)
  ) <= 25
)
SELECT pin, AVG(lat) AS lat, AVG(lon) AS lon
FROM near_reference
GROUP BY pin
HAVING COUNT(*) * 2 > MAX(point_count);
~~~~

### setup_facility_contacts.sql

~~~~sql
-- Refresh the existing facilities table with contacts from the original dataset.
CREATE OR REPLACE TABLE workspace.datathon_silver.gold_india_facilities AS
WITH supported_evidence AS (
  SELECT campus_id, evidence_text, source_url,
    ROW_NUMBER() OVER (PARTITION BY campus_id ORDER BY evidence_id) AS evidence_rank
  FROM workspace.datathon_silver.silver_service_evidence
  WHERE service = 'dialysis'
    AND evidence_status = 'supported'
    AND source_url IS NOT NULL
),
source_rows AS (
  SELECT campus_id, ids.raw_id
  FROM workspace.datathon_silver.silver_facilities
  LATERAL VIEW EXPLODE(source_ids) ids AS raw_id
),
raw_contacts AS (
  SELECT f.campus_id, TRIM(CAST(r.officialPhone AS STRING)) AS primary_phone,
    FROM_JSON(
      GET_JSON_OBJECT(TO_JSON(NAMED_STRUCT('v', r.phone_numbers)), '$.v'), 'ARRAY<STRING>'
    ) AS alternate_phones
  FROM source_rows f
  JOIN databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities r
    ON f.raw_id = LOWER(TRIM(CAST(r.unique_id AS STRING)))
),
contacts AS (
  SELECT campus_id, TRUE AS is_primary, primary_phone AS phone FROM raw_contacts
  UNION ALL
  SELECT campus_id, FALSE AS is_primary, phone
  FROM raw_contacts LATERAL VIEW EXPLODE(alternate_phones) phones AS phone
),
normalized AS (
  SELECT campus_id, is_primary, CASE
    WHEN digits RLIKE r'^1800[0-9]{6,8}$' THEN digits
    WHEN digits RLIKE r'^91[1-9][0-9]{9}$' THEN CONCAT('+', digits)
    WHEN digits RLIKE r'^0091[1-9][0-9]{9}$' THEN CONCAT('+', SUBSTRING(digits, 3))
    WHEN digits RLIKE r'^0?[1-9][0-9]{9}$' THEN CONCAT('+91', RIGHT(digits, 10))
  END AS phone
  FROM (
    SELECT campus_id, is_primary, REGEXP_REPLACE(phone, r'[^0-9]', '') AS digits
    FROM contacts
    WHERE TRIM(phone) RLIKE r'^\+?[0-9() .-]+$'
  ) digits_only
),
facility_contacts AS (
  SELECT campus_id, CASE
    WHEN COUNT(DISTINCT CASE WHEN is_primary THEN phone END) = 1
      THEN MAX(CASE WHEN is_primary THEN phone END)
    WHEN COUNT(DISTINCT phone) = 1 THEN MAX(phone)
  END AS phone
  FROM normalized
  GROUP BY campus_id
)
SELECT f.campus_id, f.name AS facility_name, f.address_line1, f.pin_clean,
       f.city, f.state, f.latitude, f.longitude,
       f.eligible_for_distance, e.evidence_text, e.source_url, c.phone, f.location_flags
FROM workspace.datathon_silver.silver_facilities f
LEFT JOIN supported_evidence e
  ON f.campus_id = e.campus_id AND e.evidence_rank = 1
LEFT JOIN facility_contacts c ON f.campus_id = c.campus_id;
~~~~

### setup_service_confidence.sql

~~~~sql
CREATE OR REPLACE TABLE workspace.datathon_silver.gold_india_service_confidence AS
WITH service_patterns AS (
  SELECT * FROM VALUES
    ('dialysis', r'\b(?:ha?emodialysis|(?:peritoneal )?dialysis)\b', r'\bnephrology\b'),
    ('mri', r'\b(?:mri|magnetic resonance imaging)\b', NULL),
    ('ct_scan', r'\b(?:ct[ -]+(?:scans?|scanners?)|computed tomography)\b', NULL),
    ('icu', r'\b(?:icu|intensive care units?)\b', NULL),
    ('nicu', r'\b(?:nicu|neonatal intensive care units?)\b', r'\bneonatology\b'),
    ('emergency_care', r'\b(?:emergency (?:department|room|services?|care|unit)|accident and emergency)\b', NULL),
    ('c_section', r'\b(?:c[ -]sections?|caesarean(?: section| delivery)?|cesarean(?: section| delivery)?)\b', NULL),
    ('cardiac_surgery', r'\b(?:cardiac surgery|heart surgery|open[ -]heart surgery|coronary artery bypass|cabg)\b', r'\b(?:cardiothoracic surgery|cardiology)\b'),
    ('burn_care', r'\b(?:burns? (?:unit|care|cent(?:er|re)))\b', NULL),
    ('chemotherapy', r'\bchemotherapy\b', NULL),
    ('radiotherapy', r'\b(?:radiotherapy|radiation therapy)\b', NULL),
    ('kidney_transplant', r'\b(?:kidney|renal) transplant(?:ation)?\b', NULL)
    AS p(service, term, related_specialty)
),
facility_ids AS (
  SELECT f.campus_id, ids.raw_id,
    COALESCE(ARRAY_CONTAINS(f.location_flags, 'conflicting_source_id_variants'), FALSE)
      OR COALESCE(ARRAY_CONTAINS(f.location_flags, 'mixed_facility_and_address'), FALSE)
      AS attribution_conflict
  FROM workspace.datathon_silver.silver_facilities f
  LATERAL VIEW EXPLODE(f.source_ids) ids AS raw_id
),
fields AS (
  SELECT f.campus_id, f.raw_id, f.attribution_conflict, fields.field_name, fields.field_value,
    GET_JSON_OBJECT(TO_JSON(NAMED_STRUCT('v', r.source_urls)), '$.v') AS source_urls_json,
    NULLIF(TRIM(REGEXP_REPLACE(LOWER(CAST(r.name AS STRING)), r'\s+', ' ')), '') AS facility_name
  FROM facility_ids f
  JOIN databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities r
    ON f.raw_id = LOWER(TRIM(CAST(r.unique_id AS STRING)))
  LATERAL VIEW STACK(5,
    'capability', GET_JSON_OBJECT(TO_JSON(NAMED_STRUCT('v', r.capability)), '$.v'),
    'procedure', GET_JSON_OBJECT(TO_JSON(NAMED_STRUCT('v', r.procedure)), '$.v'),
    'description', GET_JSON_OBJECT(TO_JSON(NAMED_STRUCT('v', r.description)), '$.v'),
    'equipment', GET_JSON_OBJECT(TO_JSON(NAMED_STRUCT('v', r.equipment)), '$.v'),
    'specialties', GET_JSON_OBJECT(TO_JSON(NAMED_STRUCT('v', r.specialties)), '$.v')
  ) fields AS field_name, field_value
),
excerpts AS (
  SELECT DISTINCT f.campus_id, f.raw_id, f.attribution_conflict, f.field_name, f.facility_name, f.source_urls_json,
    TRIM(claims.excerpt) AS excerpt
  FROM fields f
  LATERAL VIEW EXPLODE(COALESCE(
    FROM_JSON(f.field_value, 'ARRAY<STRING>'), ARRAY(f.field_value)
  )) items AS item
  LATERAL VIEW EXPLODE(SPLIT(item, r'(?<=[!?])\s+|(?<=\.)\s+(?=[A-Z])|[;\r\n]+')) claims AS excerpt
  WHERE TRIM(claims.excerpt) <> ''
),
service_text AS (
  SELECT *,
    CASE WHEN facility_name IS NULL THEN LOWER(excerpt)
      ELSE REGEXP_REPLACE(
        REGEXP_REPLACE(LOWER(excerpt), r'\s+', ' '),
        CONCAT(r'(?<![\p{L}\p{N}])\Q', facility_name, r'\E(?![\p{L}\p{N}])'),
        ' '
      )
    END AS claim_text
  FROM excerpts
),
normalized AS (
  SELECT *,
    REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(claim_text,
      r'\b(?:pet|spect)\s*[-/–]?\s*ct(?:[ -]+(?:scans?|scanners?))?\b', ' '),
      r'\b(?:neonatal|pediatric|paediatric) intensive care units?\b', ' '),
      r'\bct[ -]+(?:scans?|scanners?)\s+simulators?\b|\bct[ -]+simulation\b', ' ') AS general_text
  FROM service_text
),
candidates AS (
  SELECT n.*, p.service, p.term, p.related_specialty,
    CASE WHEN p.service IN ('ct_scan', 'icu') THEN n.general_text ELSE n.claim_text END AS claim
  FROM normalized n CROSS JOIN service_patterns p
),
classified AS (
  SELECT *,
    CASE
      WHEN attribution_conflict THEN 'attribution'
      WHEN claim RLIKE CONCAT(r'(?:\bno\s+|\bwithout\s+|\b(?:does|do|did) not (?:offer|provide|perform)\s+)', term)
        OR claim RLIKE CONCAT(term, r'\s+(?:(?:is|are)\s+)?(?:not (?:currently )?available|unavailable|not offered|not provided)\b')
        THEN 'negative'
      WHEN claim RLIKE r'\b(?:formerly|previously|historical|used to|discontinued|no longer|expired|closed permanently)\b'
        OR TRY_CAST(REGEXP_EXTRACT(claim,
          r'\b(?:expires|expired|valid until|valid through)\s+(?:on\s+)?(\d{4}-\d{2}-\d{2})\b', 1) AS DATE) < CURRENT_DATE()
        THEN 'historical'
      WHEN claim RLIKE r'\b(?:refer(?:s|red|ring)?\s+(?:patients?|to|for|elsewhere|out)|referrals?\s+(?:to|for|only)|transferred\s+(?:to|for|elsewhere))\b'
        OR claim RLIKE r'\b(?:planned|proposed|coming soon|under construction|training|workshop|course|research only)\b'
        THEN 'referral_or_planned'
      WHEN field_name = 'equipment' THEN 'equipment'
      WHEN field_name = 'specialties' THEN 'specialty'
      WHEN claim RLIKE r'\b(?:machine|scanner|equipment|simulator)s?\b' THEN 'equipment'
      WHEN claim RLIKE CONCAT(r'\b(?:offers?|provides?|performs?|conducts?|administers?|operates?|delivers?|offering|providing)\b[^.!?]{0,70}', term)
        OR claim RLIKE CONCAT(term, r'\s+(?:(?:services?|treatments?)\s+)?(?:is |are )?(?:available|offered|provided|performed|operational)\b')
        OR (field_name IN ('procedure', 'capability')
          AND claim RLIKE CONCAT(r'^\s*(?:advanced |routine |diagnostic )?', term, r'(?:\s+(?:procedures?|services?|treatments?|units?|centres?|centers?))?\s*[.!]?\s*$'))
        THEN 'direct'
      ELSE 'mention'
    END AS reason
  FROM candidates
  WHERE claim RLIKE term
     OR (field_name = 'specialties' AND claim_text RLIKE related_specialty)
),
summary AS (
  SELECT campus_id, service,
    MIN(CASE WHEN reason = 'direct' THEN NAMED_STRUCT(
      'excerpt', excerpt, 'field', field_name, 'raw_id', raw_id, 'urls', source_urls_json
    ) END) AS direct_example
  FROM classified
  GROUP BY campus_id, service
),
ranked AS (
  SELECT c.*, s.direct_example,
    CASE
      WHEN c.reason = 'negative' AND s.direct_example IS NOT NULL THEN 'conflict'
      WHEN c.reason = 'negative' THEN 'negative'
      ELSE c.reason
    END AS evidence_reason,
    ROW_NUMBER() OVER (
      PARTITION BY c.campus_id, c.service
      ORDER BY CASE c.reason
        WHEN 'attribution' THEN 0 WHEN 'negative' THEN 1 WHEN 'direct' THEN 2
        WHEN 'historical' THEN 3 WHEN 'referral_or_planned' THEN 4
        WHEN 'equipment' THEN 5 WHEN 'specialty' THEN 6 ELSE 7 END,
        CASE c.field_name WHEN 'procedure' THEN 0 WHEN 'capability' THEN 1
          WHEN 'description' THEN 2 WHEN 'equipment' THEN 3 ELSE 4 END,
        LENGTH(c.excerpt), c.excerpt, c.raw_id, c.source_urls_json
    ) AS evidence_rank
  FROM classified c
  JOIN summary s ON c.campus_id = s.campus_id AND c.service = s.service
)
SELECT campus_id, service,
  CASE WHEN evidence_reason = 'direct' THEN 3
    WHEN evidence_reason IN ('equipment', 'specialty', 'mention') THEN 2 ELSE 1 END AS confidence,
  evidence_reason,
  field_name AS source_field,
  excerpt AS evidence_text,
  raw_id AS source_record_id,
  source_urls_json,
  CASE WHEN evidence_reason = 'conflict' THEN direct_example.excerpt END AS counter_evidence_text,
  CASE WHEN evidence_reason = 'conflict' THEN direct_example.field END AS counter_source_field,
  CASE WHEN evidence_reason = 'conflict' THEN direct_example.raw_id END AS counter_record_id,
  CASE WHEN evidence_reason = 'conflict' THEN direct_example.urls END AS counter_source_urls_json
FROM ranked
WHERE evidence_rank = 1;
~~~~

## Appendix B — Databricks app source

The demo above contains the complete v22 `search.sql`. The remaining text source files are listed below. To deploy the app, keep their original filenames and include `favicon.png` from the v22 deployment package.

The app uses the `sql-warehouse` resource to query the same Gold tables as the notebook demo.

### app.py

~~~~python
from base64 import b64encode
from contextlib import contextmanager
from html import escape
from itertools import groupby
import json
import logging
import math
import os
from pathlib import Path
import re
from urllib.parse import urlsplit

from databricks import sql
from databricks.sdk.core import Config
import streamlit as st

from locale_strings import FIELDS, MESSAGES, SERVICES

DISTANCE_HALF_LIFE_KM = 25
RANKINGS = ("overall", "nearest", "confidence")
RANKING_CSS = Path(__file__).with_name("ranking.css").read_text(encoding="utf-8")
SEARCH_SQL = Path(__file__).with_name("search.sql").read_text(encoding="utf-8")
PIN_LOOKUP_SQL = Path(__file__).with_name("pin_lookup.sql").read_text(encoding="utf-8")
MAP_PIN_URL = "data:image/png;base64," + b64encode(
    Path(__file__).with_name("favicon.png").read_bytes()
).decode("ascii")
MAP_PICKER = st.components.v2.component(
    "india_pin_picker",
    html="""
        <link rel="stylesheet" class="picker-leaflet-css">
        <div class="picker-frame">
            <div class="picker-map" role="region"></div>
            <img class="picker-pin" alt="" aria-hidden="true">
        </div>
        <div class="picker-actions">
            <span class="picker-status" role="status"></span>
            <button type="button" class="picker-confirm" disabled></button>
        </div>
    """,
    css=Path(__file__).with_name("map_picker.css").read_text(encoding="utf-8"),
    js=Path(__file__).with_name("map_picker.js").read_text(encoding="utf-8"),
)
TRANSLATE_SQL = """
WITH originals AS (
    SELECT DISTINCT EXPLODE(FROM_JSON(:texts, 'ARRAY<STRING>')) AS original
)
SELECT original, ai_translate(original, 'hi') AS hindi
FROM originals
"""


@contextmanager
def warehouse_cursor():
    warehouse_id = os.environ["DATABRICKS_WAREHOUSE_ID"]
    config = Config()
    with sql.connect(
        server_hostname=urlsplit(config.host).hostname,
        http_path=f"/sql/1.0/warehouses/{warehouse_id}",
        credentials_provider=lambda: config.authenticate,
    ) as connection:
        with connection.cursor() as cursor:
            yield cursor


def search_facilities(pin, services, origin):
    latitude, longitude = origin if origin is not None else (0.0, 0.0)
    with warehouse_cursor() as cursor:
        cursor.execute(
            SEARCH_SQL,
            parameters={
                "pin": pin,
                "use_map_point": int(origin is not None),
                "origin_lat": latitude,
                "origin_lon": longitude,
                "services": json.dumps(services),
                "distance_half_life_km": DISTANCE_HALF_LIFE_KM,
            },
        )
        return cursor.fetchall()


def find_pin(latitude, longitude):
    with warehouse_cursor() as cursor:
        cursor.execute(PIN_LOOKUP_SQL, parameters={"latitude": latitude, "longitude": longitude})
        return cursor.fetchone()


def map_labels(tr, title):
    return {
        "title": title,
        "pinIcon": MAP_PIN_URL,
        "loading": tr("map_loading"),
        "loadError": tr("map_load_error"),
        "tilesError": tr("map_tile_error"),
        "zoomIn": tr("map_zoom_in"),
        "zoomOut": tr("map_zoom_out"),
    }


def show_results_map(facilities, translations, tr, from_map, excluded_location_count):
    @st.dialog(tr("results_map_title"), width="large")
    def dialog():
        st.caption(tr("results_map_note"))
        if excluded_location_count:
            st.caption(tr("location_excluded", count=excluded_location_count))
        origin_label = tr("results_map_origin_pin" if from_map else "results_map_origin_postal")
        st.caption(tr("results_map_legend", origin=origin_label))
        first = facilities[0][0]
        origin = {
            "lat": float(first.origin_latitude),
            "lon": float(first.origin_longitude),
            "label": origin_label,
        }
        markers = [
            {
                "lat": float(row.latitude),
                "lon": float(row.longitude),
                "rank": int(row.result_rank),
                "name": translations.get(row.facility_name, row.facility_name),
                "distance": tr("distance", km=row.straight_line_km),
                "locationWarning": tr("location_warning") if row.location_flags else "",
            }
            for row in (facility[0] for facility in facilities)
        ]
        MAP_PICKER(
            key="facility_results_map",
            data={**map_labels(tr, tr("results_map_title")), "markers": markers, "origin": origin},
        )

    dialog()


def pick_pin_on_map(tr):
    @st.dialog(tr("map_title"), width="large")
    def dialog():
        st.caption(tr("map_help"))
        picker = MAP_PICKER(
            key="india_pin_map",
            data={
                **map_labels(tr, tr("map_title")),
                "confirm": tr("map_confirm"),
                "zoomForPin": tr("map_zoom_hint"),
                "checking": tr("map_checking"),
                "start": st.session_state.get("map_view"),
            },
        )
        point = getattr(picker, "confirmed", None)
        if point is None:
            return

        try:
            latitude, longitude, zoom = float(point["lat"]), float(point["lon"]), int(point["zoom"])
            if not (
                math.isfinite(latitude) and math.isfinite(longitude)
                and 11 <= zoom <= 19 and -90 <= latitude <= 90 and -180 <= longitude <= 180
            ):
                raise ValueError("Invalid map point")
        except (TypeError, KeyError, ValueError, OverflowError):
            st.warning(tr("map_invalid_point"))
            return

        st.session_state["map_view"] = {"lat": latitude, "lon": longitude, "zoom": zoom}
        match = None
        try:
            with st.spinner(tr("map_checking")):
                match = find_pin(latitude, longitude)
        except Exception:
            logging.exception("Optional map PIN lookup failed")
        pin = str(match.pin) if match else ""
        st.session_state["pending_map_pin"] = (pin, latitude, longitude)
        st.rerun()

    dialog()


def translate_details(rows):
    originals = {
        text
        for row in rows
        for text in (row.facility_name, row.address_line1, row.city, row.state, row.evidence_text, row.counter_evidence_text)
        if text
    }
    if not originals:
        return {}, False
    try:
        with warehouse_cursor() as cursor:
            cursor.execute(TRANSLATE_SQL, parameters={"texts": json.dumps(sorted(originals))})
            translations = {
                row.original: row.hindi.strip()
                for row in cursor.fetchall()
                if row.hindi and row.hindi.strip()
            }
    except Exception:
        logging.exception("Hindi translation of facility details failed")
        return {}, True
    return translations, bool(originals.difference(translations))


def render():
    hindi = st.session_state.get("language") == "हिन्दी"

    def tr(key, **values):
        return MESSAGES[key][hindi].format(**values)

    def service_label(service):
        return SERVICES[service][hindi]

    logo_path = str(Path(__file__).with_name("favicon.png"))
    st.set_page_config(
        page_title=tr("title"),
        page_icon=logo_path,
        layout="centered",
    )
    pending_pin = st.session_state.pop("pending_map_pin", None)
    if pending_pin is not None:
        st.session_state["pin"] = pending_pin[0]
        st.session_state["mapped_pin"] = pending_pin
        st.session_state.pop("search_result", None)

    _, language_column = st.columns([3, 1])
    with language_column:
        st.selectbox("Language / भाषा", ("English", "हिन्दी"), key="language", label_visibility="collapsed")
    with st.container(key="page_hero"):
        with st.container(horizontal=True, horizontal_alignment="center"):
            st.html(
                     f'<img src="{MAP_PIN_URL}" alt="" '
                     'width="76" height="76" style="display:block;margin:auto;">'
                   )
        st.title(tr("title"), text_alignment="center")
        st.caption(tr("intro"), text_alignment="center")

    with st.container(border=True):
        with st.container(horizontal=True, horizontal_alignment="right", vertical_alignment="bottom"):
            pin = st.text_input(
                tr("pin"), placeholder=tr("pin_example"), key="pin", width="stretch"
            ).strip()
            open_map = st.button(tr("open_map"), icon=":material/location_on:", width="content")
        mapped_pin = st.session_state.get("mapped_pin")
        origin = None
        if mapped_pin is not None and pin == mapped_pin[0]:
            origin = mapped_pin[1:]
            st.caption(tr("map_pin_filled", pin=pin) if pin else tr("map_without_pin"))
        elif mapped_pin is not None:
            st.session_state.pop("mapped_pin", None)
        services = st.multiselect(
            tr("services"),
            options=list(SERVICES),
            format_func=service_label,
            placeholder=tr("choose"),
            help=tr("help"),
            select_all=False,
            key="services",
        )
        submit = st.button(tr("search"), type="primary")

    if open_map:
        pick_pin_on_map(tr)

    if submit:
        st.session_state.pop("search_result", None)
        st.session_state["ranking_mode"] = "overall"
        if origin is None and not re.fullmatch(r"[1-9][0-9]{5}", pin):
            st.session_state["search_result"] = {"notice": "invalid_pin"}
        elif not services:
            st.session_state["search_result"] = {"notice": "no_services"}
        else:
            try:
                with st.spinner(tr("searching")):
                    rows = search_facilities(pin, services, origin)
                st.session_state["search_result"] = {"pin": pin, "from_map": origin is not None, "services": services, "rows": rows}
            except Exception:
                logging.exception("Facility lookup failed")
                st.session_state["search_result"] = {"notice": "search_failed"}

    render_results(st.session_state.get("search_result"), tr, service_label, hindi)
    st.html(
        '<footer style="border-top:1px solid var(--st-border-color, #d8d4d0);'
        'padding-top:.65rem;text-align:center;font-size:.85rem;opacity:.8">'
        f'{escape(tr("page_credit"))}</footer>'
    )


def render_results(result, tr, service_label, hindi):
    if result is None:
        st.html("""
            <style>
            .stMainBlockContainer:has(.st-key-page_hero) {
                padding-block-start: 4.25rem;
                padding-block-end: 1rem;
            }
            .st-key-page_hero [data-testid="stVerticalBlock"] {
                gap: 0.35rem;
            }
            .st-key-page_hero h1 {
                padding-block: 0.1rem 0.3rem;
                font-size: clamp(1.6rem, 4.2vw, 2.65rem);
                line-height: 1.12;
            }
            </style>
        """)
        return
    notice = result.get("notice")
    if notice:
        show = st.error if notice == "search_failed" else st.warning
        show(tr(notice))
        return

    rows = result["rows"]
    if not rows:
        st.info(tr("no_results"))
        return
    first = rows[0]
    if first.status == "unknown_pin":
        st.info(tr("unknown_pin"))
        return

    excluded_location_count = first.excluded_location_count
    if excluded_location_count:
        st.caption(tr("location_excluded", count=excluded_location_count))

    searched_services = result["services"]
    if first.status == "no_matches":
        st.info(tr("no_service_records"))
        return

    unavailable = set(json.loads(first.unavailable_services_json))
    if unavailable:
        labels = ", ".join(service_label(service) for service in searched_services if service in unavailable)
        st.warning(tr("unsupported_services", services=labels))

    translations = {}
    if hindi:
        if "translations" not in result:
            with st.spinner(tr("translating")):
                result["translations"] = translate_details(rows)
        translations, incomplete = result["translations"]
        if translations:
            st.caption(tr("auto_translation"))
        if incomplete:
            st.caption(tr("translation_missing"))

    st.html(f"<style>{RANKING_CSS}</style>")
    st.session_state.setdefault("ranking_mode", "overall")
    heading, map_column = st.columns([3, 2], vertical_alignment="center")
    heading.subheader(tr("results_heading"))
    with map_column, st.container(horizontal=True, horizontal_alignment="right"):
        has_map_results = any(row.ranking == st.session_state["ranking_mode"] for row in rows)
        open_results_map = st.button(
            tr("results_map_button"), icon=":material/map:", key="results_map", disabled=not has_map_results
        )
    ranking = st.segmented_control(
        tr("ranking_label"), RANKINGS,
        format_func=lambda key: tr(f"ranking_{key}"),
        selection_mode="single", required=True, key="ranking_mode",
    )
    origin_label = "distance_origin_map" if result["from_map"] else "distance_origin"
    st.caption(tr(origin_label, pin=result["pin"]))
    st.caption(tr(f"ranking_note_{ranking}"))
    ranked_rows = (row for row in rows if row.ranking == ranking)
    facilities = [list(evidence) for _, evidence in groupby(ranked_rows, key=lambda row: row.result_rank)]
    if not facilities:
        st.warning(tr("no_recommendations"))
    if open_results_map and facilities:
        show_results_map(facilities, translations, tr, result["from_map"], excluded_location_count)
    for facility in facilities:
        render_facility(facility, searched_services, hindi, translations, tr, service_label, ranking)
    st.caption(tr("confirm"))


def render_facility(evidence_rows, services, hindi, translations, tr, service_label, ranking):
    row = evidence_rows[0]
    evidence_by_service = {evidence.service: evidence for evidence in evidence_rows}

    with st.container(border=True):
        name = translations.get(row.facility_name, row.facility_name)
        rank = row.result_rank
        positive_count = row.high_count + row.medium_count
        medal = {1: "gold", 2: "silver", 3: "bronze"}.get(rank) if positive_count else None
        rank_label = tr(f"rank_{medal}") if medal else tr("rank_label", rank=rank)
        badge_class = f" rank-badge--{medal}" if medal else ""
        warning = ""
        if ranking == "nearest" and positive_count < len(services):
            warning_key = "rank_warning_partial" if positive_count else "rank_warning_low"
            label = escape(tr(warning_key))
            warning = (
                f' <span class="rank-evidence-warning" role="img" '
                f'aria-label="{label}" title="{label}">!</span>'
            )
        st.html(
            f'<div class="facility-rank-heading">'
            f'<span class="rank-badge{badge_class}" role="img" aria-label="{escape(rank_label)}">'
            f'<span class="rank-badge__number" aria-hidden="true">{rank}</span></span>'
            f'<h3>{escape(name)}{warning}</h3></div>'
        )
        address_parts = (row.address_line1, row.city, row.state, row.pin_clean)
        address = ", ".join(translations.get(part, part) for part in address_parts if part)
        if address:
            st.text(address)
        if row.location_flags:
            st.warning(tr("location_warning"))
        if row.phone:
            st.text(tr("facility_phone", phone=row.phone))
        distance = tr("distance", km=row.straight_line_km)
        if ranking == "overall":
            evidence_points = round(float(row.evidence_points), 1)
            distance_points = round(float(row.distance_points), 1)
            score = tr(
                "overall_breakdown", score=evidence_points + distance_points,
                evidence=evidence_points, distance=distance_points,
            )
            st.caption(f"{distance} · {score}")
        elif ranking == "confidence":
            st.caption(f'{distance} · {tr("confidence_score", score=100 * row.confidence_score)}')
        else:
            st.caption(distance)
        st.caption(tr("confidence_counts", high=row.high_count, medium=row.medium_count,
                      low=len(services) - row.high_count - row.medium_count))
        if len(services) > 1:
            supported = [service_label(service) for service in services
                         if evidence_by_service[service].confidence >= 2]
            uncertain = [service_label(service) for service in services
                         if evidence_by_service[service].confidence == 1]
            if supported:
                st.caption(tr("selected_supported", count=len(supported), total=len(services),
                              services=", ".join(supported)))
            if uncertain:
                st.caption(tr("selected_uncertain", services=", ".join(uncertain)))

        for service in services:
            evidence = evidence_by_service[service]
            color = {3: "#15803d", 2: "#b77905", 1: "#c24148"}[evidence.confidence]
            label = tr(f"confidence_{evidence.confidence}")
            reason = tr(f"reason_{evidence.evidence_reason}")
            st.html(
                f'<div style="border:1px solid color-mix(in srgb, {color} 60%, transparent);'
                f'border-left:4px solid {color};padding:.65rem .8rem;border-radius:.6rem">'
                f'<strong>{escape(service_label(service))}</strong> · {escape(label)}'
                f'<div>{escape(reason)}</div></div>'
            )
            if evidence.evidence_text:
                with st.expander(tr("view_evidence", service=service_label(service))):
                    render_excerpt(evidence.evidence_text, evidence.source_field,
                                   evidence.source_record_id, evidence.source_urls_json, hindi, translations, tr)
                    if evidence.counter_evidence_text:
                        st.caption(tr("counter_evidence"))
                        render_excerpt(evidence.counter_evidence_text, evidence.counter_source_field,
                                       evidence.counter_record_id, evidence.counter_source_urls_json,
                                       hindi, translations, tr)


def render_excerpt(text, field, source_id, source_urls_json, hindi, translations, tr):
    label = FIELDS.get(field, field) if hindi else field
    st.caption(tr("record_source", field=label, id=source_id))
    if text in translations and translations[text] != text:
        st.text(translations[text])
        st.caption(tr("original_excerpt"))
    st.text(text)
    urls = record_urls(source_urls_json)
    if urls:
        st.caption(tr("source_links"))
        st.html("<ul>" + "".join(
            f'<li><a href="{escape(url)}" target="_blank" rel="noopener noreferrer">'
            f'{escape(urlsplit(url).netloc + urlsplit(url).path)}</a></li>'
            for url in urls
        ) + "</ul>")
        st.caption(tr("source_link_note"))


def record_urls(value):
    if not value:
        return []
    try:
        candidates = json.loads(value)
    except ValueError:
        candidates = value
    if isinstance(candidates, str):
        candidates = [candidates]
    if not isinstance(candidates, list):
        return []

    links = []
    for candidate in candidates:
        if not isinstance(candidate, str):
            continue
        url = candidate.strip()
        if not url or re.search(r"[\s\\\x00-\x1f\x7f]", url):
            continue
        try:
            parts = urlsplit(url)
            if parts.scheme not in ("http", "https") or not parts.hostname or parts.username is not None:
                continue
        except ValueError:
            continue
        if url not in links:
            links.append(url)
    return links


if __name__ == "__main__":
    render()
~~~~

### locale_strings.py

~~~~python
SERVICES = {
    "dialysis": ("Dialysis", "डायलिसिस"),
    "emergency_care": ("Emergency care", "आपातकालीन देखभाल"),
    "icu": ("ICU", "गहन चिकित्सा इकाई (आईसीयू)"),
    "ct_scan": ("CT scan", "सीटी स्कैन"),
    "mri": ("MRI", "एमआरआई"),
    "nicu": ("NICU", "नवजात गहन चिकित्सा इकाई (एनआईसीयू)"),
    "kidney_transplant": ("Kidney transplant", "गुर्दा प्रत्यारोपण"),
    "c_section": ("C-section", "सिज़ेरियन प्रसव"),
    "cardiac_surgery": ("Cardiac surgery", "हृदय शल्य चिकित्सा"),
    "chemotherapy": ("Chemotherapy", "कीमोथेरेपी"),
    "radiotherapy": ("Radiotherapy", "रेडियोथेरेपी"),
    "burn_care": ("Burn care", "जलने की चोटों का उपचार"),
}

FIELDS = {
    "capability": "उपचार सुविधाएँ",
    "procedure": "चिकित्सा प्रक्रियाएँ",
    "description": "विवरण",
    "equipment": "चिकित्सा उपकरण",
    "specialties": "चिकित्सा विशेषज्ञताएँ",
}

MESSAGES = {
    "title": ("Find Your Nearest Healthcare", "अपने निकटतम स्वास्थ्य सेवा खोजें"),
    "intro": (
        "Search by Indian PIN code or map and check the evidence for each selected service.",
        "भारतीय पिन कोड के अनुसार खोजें और प्रत्येक चयनित सेवा के लिए साक्ष्य जांचें।",
    ),
    "pin": ("Address (PIN code)", "पता (पिन कोड)"),
    "pin_example": ("e.g. 226003", "उदाहरण: 226003"),
    "open_map": ("Choose on map", "नक्शे पर चुनें"),
    "map_title": ("Choose a location", "स्थान चुनें"),
    "map_help": (
        "Drag or zoom the map until the red pin is over your location, then confirm.",
        "नक्शे को खिसकाएँ या ज़ूम करें। लाल पिन को अपनी जगह पर लाकर पुष्टि करें।",
    ),
    "map_confirm": ("Confirm location", "स्थान की पुष्टि करें"),
    "map_loading": ("Loading map…", "नक्शा लोड हो रहा है…"),
    "map_load_error": ("The map could not load. You can still enter a PIN.", "नक्शा लोड नहीं हो सका। आप पिन कोड डाल सकते हैं।"),
    "map_tile_error": ("Some map imagery did not load.", "नक्शे के कुछ हिस्से लोड नहीं हो सके।"),
    "map_zoom_hint": ("Zoom in to confirm a location.", "स्थान की पुष्टि के लिए ज़ूम इन करें।"),
    "map_zoom_in": ("Zoom in", "ज़ूम इन करें"),
    "map_zoom_out": ("Zoom out", "ज़ूम आउट करें"),
    "map_checking": ("Looking up the nearest PIN…", "सबसे नज़दीकी पिन कोड खोजा जा रहा है…"),
    "map_invalid_point": (
        "This point is invalid. Choose a location on the map again.",
        "यह स्थान मान्य नहीं है। नक्शे पर दोबारा जगह चुनें।",
    ),
    "map_pin_filled": (
        "Map point selected · Estimated PIN {pin}. Edit the PIN to use its postal center instead.",
        "नक्शे पर स्थान चुना गया · अनुमानित पिन {pin}। डाक केंद्र से दूरी मापने के लिए पिन बदलें।",
    ),
    "map_without_pin": (
        "Map point selected. You can search from this point without a PIN code.",
        "नक्शे पर स्थान चुना गया। आप पिन कोड के बिना इस स्थान से खोज सकते हैं।",
    ),
    "distance_origin_map": (
        "Straight-line distances from your selected point on the map.",
        "नक्शे पर आपके चुने हुए स्थान से सीधी दूरी।",
    ),
    "results_map_button": ("View map", "नक्शा देखें"),
    "results_map_title": ("Facility locations", "स्वास्थ्य केंद्रों के स्थान"),
    "results_map_note": (
        "Markers match this ranking and use the facility coordinates in the dataset. Select a marker to see its name.",
        "निशान इसी क्रम से मेल खाते हैं और डेटा में दर्ज केंद्रों के निर्देशांक दिखाते हैं। नाम देखने के लिए निशान चुनें।",
    ),
    "results_map_origin_pin": ("Your map pin", "नक्शे पर आपका चुना हुआ स्थान"),
    "results_map_origin_postal": ("Approximate PIN code center", "पिन कोड का अनुमानित केंद्र"),
    "results_map_legend": (
        "Medical pin: {origin} · Red numbered circles: facilities",
        "चिकित्सा पिन: {origin} · लाल क्रमांक वाले गोले: स्वास्थ्य केंद्र",
    ),
    "facility_phone": ("Listed phone: {phone}", "सूचीबद्ध फ़ोन: {phone}"),
    "services": ("Services needed", "ज़रूरी सेवाएँ"),
    "choose": ("Choose one or more services", "एक या अधिक सेवाएँ चुनें"),
    "help": (
        "Compare the strength of the dataset evidence for each requested service.",
        "चुनी गई हर सेवा के लिए डेटा में उपलब्ध साक्ष्य की तुलना करें।",
    ),
    "search": ("Find facilities", "स्वास्थ्य सुविधाएँ खोजें"),
    "invalid_pin": ("Enter a valid six-digit Indian PIN code.", "भारत का मान्य छह अंकों वाला पिन कोड डालें।"),
    "no_services": ("Select at least one service.", "कम से कम एक सेवा चुनें।"),
    "searching": ("Searching facility records…", "स्वास्थ्य केंद्रों के रिकॉर्ड खोजे जा रहे हैं…"),
    "search_failed": (
        "The search could not connect to the data. Please try again.",
        "खोज के लिए डेटा उपलब्ध नहीं हो सका। कृपया फिर से कोशिश करें।",
    ),
    "no_results": ("No results for this search.", "इस खोज का कोई परिणाम नहीं मिला।"),
    "unknown_pin": (
        "No reliable search point was found for this PIN.",
        "इस पिन कोड के लिए खोज का विश्वसनीय स्थान नहीं मिला।",
    ),
    "distance_origin": (
        "Distances are measured in a straight line from the approximate center of PIN {pin}. No distance limit applies.",
        "दूरियाँ पिन कोड {pin} के अनुमानित केंद्र से सीधी रेखा में मापी गई हैं। दूरी की कोई सीमा नहीं है।",
    ),
    "results_heading": ("Search results", "खोज के परिणाम"),
    "ranking_label": ("Sort results", "परिणामों का क्रम"),
    "ranking_overall": ("Overall", "समग्र"),
    "ranking_nearest": ("Nearest first", "सबसे नज़दीक"),
    "ranking_confidence": ("Confidence first", "विश्वास पहले"),
    "ranking_note_overall": (
        "Up to 10 facilities · 60% evidence strength · 40% distance.",
        "अधिकतम 10 केंद्र · 60% साक्ष्य की मज़बूती · 40% दूरी।",
    ),
    "ranking_note_nearest": (
        "Up to 5 facilities · shortest distance first.",
        "अधिकतम 5 केंद्र · सबसे कम दूरी पहले।",
    ),
    "ranking_note_confidence": (
        "Up to 5 facilities · strongest evidence first, then shortest distance.",
        "अधिकतम 5 केंद्र · पहले सबसे मज़बूत साक्ष्य, फिर सबसे कम दूरी।",
    ),
    "rank_gold": ("Rank 1 · Gold medal", "स्थान 1 · स्वर्ण पदक"),
    "rank_silver": ("Rank 2 · Silver medal", "स्थान 2 · रजत पदक"),
    "rank_bronze": ("Rank 3 · Bronze medal", "स्थान 3 · कांस्य पदक"),
    "rank_label": ("Rank {rank}", "स्थान {rank}"),
    "overall_breakdown": (
        "Overall score {score:.1f}/100 (Evidence {evidence:.1f}/60 + Distance {distance:.1f}/40)",
        "समग्र स्कोर {score:.1f}/100 (साक्ष्य {evidence:.1f}/60 + दूरी {distance:.1f}/40)",
    ),
    "rank_warning_low": (
        "Only low-confidence evidence for the selected services.",
        "चुनी गई सेवाओं के लिए केवल कम विश्वास वाला साक्ष्य है।",
    ),
    "rank_warning_partial": (
        "At least one selected service has insufficient evidence.",
        "कम से कम एक चुनी गई सेवा के लिए पर्याप्त साक्ष्य नहीं है।",
    ),
    "no_recommendations": (
        "The dataset does not contain enough positive evidence to recommend a facility for the selected services.",
        "चुनी गई सेवाओं के लिए किसी केंद्र की सिफ़ारिश करने लायक पर्याप्त सकारात्मक साक्ष्य डेटा में नहीं है।",
    ),
    "selected_supported": (
        "{count}/{total} selected services have high or medium evidence: {services}.",
        "चुनी गई {total} में से {count} सेवाओं के लिए उच्च या मध्यम साक्ष्य हैं: {services}।",
    ),
    "selected_uncertain": (
        "Insufficient evidence for: {services}.",
        "इन सेवाओं के लिए पर्याप्त साक्ष्य नहीं हैं: {services}।",
    ),
    "confidence_score": ("Evidence score {score:.1f}/100", "साक्ष्य स्कोर {score:.1f}/100"),
    "no_service_records": (
        "No facility with a usable location has a record for the selected services in the dataset.",
        "डेटा में चुनी गई सेवाओं के लिए उपयोग योग्य स्थान वाला कोई केंद्र नहीं मिला।",
    ),
    "unsupported_services": (
        "No high- or medium-confidence evidence in the searchable dataset for: {services}.",
        "खोज योग्य डेटा में इन सेवाओं के लिए उच्च या मध्यम स्तर का साक्ष्य नहीं मिला: {services}।",
    ),
    "distance": ("{km:.1f} km", "{km:.1f} किमी"),
    "confidence_counts": ("{high} high · {medium} medium · {low} low", "{high} उच्च · {medium} मध्यम · {low} कम"),
    "confidence_3": ("High confidence", "उच्च विश्वास"),
    "confidence_2": ("Medium confidence", "मध्यम विश्वास"),
    "confidence_1": ("Low confidence", "कम विश्वास"),
    "reason_direct": ("The dataset directly lists this as a service at the facility.", "डेटा में इस केंद्र की सेवा सीधे दर्ज है।"),
    "reason_equipment": ("The record describes related equipment.", "रिकॉर्ड में संबंधित उपकरण का वर्णन है।"),
    "reason_specialty": ("The record mentions a related specialty.", "रिकॉर्ड में संबंधित विशेषज्ञता का उल्लेख है।"),
    "reason_mention": ("The text mentions the service without clearly saying it is provided here.", "सेवा का उल्लेख है, पर यहाँ उपलब्धता स्पष्ट नहीं है।"),
    "reason_referral_or_planned": ("The record describes a referral, planned service or training.", "रिकॉर्ड में रेफ़रल, प्रस्तावित सेवा या प्रशिक्षण का उल्लेख है।"),
    "reason_historical": ("The record describes a past service or expired listing.", "रिकॉर्ड में पुरानी सेवा या समाप्त हो चुकी सूची का उल्लेख है।"),
    "reason_negative": ("The record says the service is unavailable or not offered.", "रिकॉर्ड के अनुसार यह सेवा उपलब्ध नहीं है।"),
    "reason_conflict": ("Dataset excerpts give conflicting information about availability.", "सेवा की उपलब्धता पर डेटा के अंश एक-दूसरे से मेल नहीं खाते।"),
    "reason_attribution": ("Conflicting facility details make the campus attribution uncertain.", "केंद्र के विवरणों में विरोध होने से सेवा किस परिसर में है, यह स्पष्ट नहीं है।"),
    "reason_missing": ("No relevant evidence was found in the supplied dataset for this campus.", "इस परिसर के लिए दिए गए डेटा में संबंधित साक्ष्य नहीं मिला।"),
    "view_evidence": ("View dataset evidence · {service}", "डेटा का साक्ष्य देखें · {service}"),
    "source_links": ("Links listed in this dataset record", "इस डेटा रिकॉर्ड में दिए गए लिंक"),
    "source_link_note": (
        "The dataset lists these links for the whole record. It does not identify which link supports each excerpt.",
        "ये लिंक पूरे रिकॉर्ड के लिए दिए गए हैं। डेटा में हर अंश का अलग स्रोत नहीं बताया गया है।",
    ),
    "location_excluded": (
        "Matching facility records excluded across India due to location conflicts: {count}.",
        "स्थान संबंधी विसंगतियों के कारण भारत भर में हटाए गए मेल खाते सुविधा रिकॉर्ड: {count}।",
    ),
    "location_warning": (
        "The dataset contains conflicting location information. Confirm the campus address before travelling.",
        "डेटा में स्थान संबंधी जानकारी मेल नहीं खाती। जाने से पहले परिसर का पता सत्यापित करें।",
    ),
    "page_credit": (
        "Data provided by Virtue Foundation · Built on Databricks",
        "डेटा: Virtue Foundation · Databricks पर बनाया गया",
    ),
    "record_source": ("Dataset field: {field} · Record ID: {id}", "डेटा का अनुभाग: {field} · रिकॉर्ड आईडी: {id}"),
    "counter_evidence": ("Conflicting excerpt", "विरोधी अंश"),
    "original_excerpt": ("Original dataset text", "डेटा का मूल पाठ"),
    "confirm": (
        "Contact the facility to confirm availability before visiting.",
        "जाने से पहले केंद्र से संपर्क करके सेवा की उपलब्धता की पुष्टि करें।",
    ),
    "translating": ("Translating facility details…", "केंद्रों के विवरण का अनुवाद हो रहा है…"),
    "auto_translation": ("Facility details are machine translated.", "केंद्रों के विवरण का मशीन से अनुवाद किया गया है।"),
    "translation_missing": (
        "Some facility details could not be translated and are shown in their original language.",
        "केंद्रों के कुछ विवरणों का अनुवाद उपलब्ध नहीं है। वे मूल भाषा में दिखाए गए हैं।",
    ),
}
~~~~

### pin_lookup.sql

~~~~sql
WITH nearby AS (
  SELECT pin, lat, lon
  FROM workspace.datathon_silver.gold_india_pin_centers
  WHERE lat BETWEEN :latitude - 0.25 AND :latitude + 0.25
    AND lon BETWEEN :longitude - 0.25 AND :longitude + 0.25
),
distances AS (
  SELECT pin, 2 * 6371 * ASIN(LEAST(1.0, SQRT(
    POWER(SIN(RADIANS(lat - :latitude) / 2), 2)
    + COS(RADIANS(:latitude)) * COS(RADIANS(lat))
      * POWER(SIN(RADIANS(lon - :longitude) / 2), 2)
  ))) AS distance_km
  FROM nearby
)
SELECT pin, distance_km
FROM distances
WHERE distance_km <= 15
ORDER BY distance_km, pin
LIMIT 1;
~~~~

### map_picker.js

~~~~javascript
const leafletUrl = "https://unpkg.com/leaflet@1.9.4/dist/leaflet.js";
const leafletCssUrl = "https://unpkg.com/leaflet@1.9.4/dist/leaflet.css";
let leafletPromise;

function loadLeaflet() {
  if (window.L?.version === "1.9.4") return Promise.resolve(window.L);
  if (!leafletPromise) {
    leafletPromise = new Promise((resolve, reject) => {
      const script = document.createElement("script");
      script.src = leafletUrl;
      script.integrity = "sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=";
      script.crossOrigin = "anonymous";
      script.onload = () => window.L ? resolve(window.L) : reject(new Error("Leaflet unavailable"));
      script.onerror = () => {
        script.remove();
        reject(new Error("Leaflet failed to load"));
      };
      document.head.append(script);
    }).catch((error) => {
      leafletPromise = undefined;
      throw error;
    });
  }
  return leafletPromise;
}

function showFacilityMarkers(L, map, markers, origin, pinIcon) {
  const locations = new Map();
  for (const marker of markers) {
    const key = `${marker.lat},${marker.lon}`;
    if (!locations.has(key)) locations.set(key, []);
    locations.get(key).push(marker);
  }

  const points = [];
  for (const group of locations.values()) {
    const point = [group[0].lat, group[0].lon];
    points.push(point);
    const ranks = group.map(item => Number(item.rank)).join(" / ");
    const popup = document.createElement("div");
    for (const item of group) {
      const entry = document.createElement("div");
      const title = document.createElement("strong");
      title.textContent = `${item.rank}. ${item.name}`;
      const distance = document.createElement("div");
      distance.textContent = item.distance;
      entry.append(title, distance);
      if (item.locationWarning) {
        const warning = document.createElement("div");
        warning.className = "facility-location-warning";
        warning.textContent = item.locationWarning;
        entry.append(warning);
      }
      popup.append(entry);
    }
    L.marker(point, {
      title: group.map(item => item.name).join(" / "),
      icon: L.divIcon({
        className: "facility-map-marker",
        html: ranks,
        iconSize: [32, 32],
        iconAnchor: [16, 16],
        popupAnchor: [0, -14],
      }),
    }).addTo(map).bindPopup(popup);
  }

  const originPoint = [origin.lat, origin.lon];
  const originLabel = document.createElement("strong");
  originLabel.textContent = origin.label;
  L.marker(originPoint, {
    title: origin.label,
    alt: origin.label,
    zIndexOffset: 1000,
    icon: L.icon({
      iconUrl: pinIcon,
      className: "search-origin-pin",
      iconSize: [52, 52],
      iconAnchor: [26, 46],
      popupAnchor: [0, -39],
    }),
  }).addTo(map).bindPopup(originLabel);
  points.push(originPoint);
  map.fitBounds(points, { padding: [36, 36], maxZoom: 14 });
}

export default function ({ parentElement, data, setTriggerValue }) {
  const frame = parentElement.querySelector(".picker-map");
  const button = parentElement.querySelector(".picker-confirm");
  const status = parentElement.querySelector(".picker-status");
  const css = parentElement.querySelector(".picker-leaflet-css");
  css.integrity = "sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY=";
  css.crossOrigin = "anonymous";
  css.href = leafletCssUrl;
  const resultsMap = Array.isArray(data.markers);
  const pickerPin = parentElement.querySelector(".picker-pin");
  pickerPin.hidden = resultsMap;
  if (!resultsMap) pickerPin.src = data.pinIcon;
  button.hidden = resultsMap;
  button.textContent = resultsMap ? "" : data.confirm;
  button.disabled = true;
  frame.setAttribute("aria-label", data.title);
  status.textContent = data.loading;
  let map;
  let closed = false;

  loadLeaflet().then((L) => {
    if (closed) return;
    const start = data.start || { lat: 22.5, lon: 79.0, zoom: 5 };
    map = L.map(frame, { zoomControl: false, minZoom: 4, maxZoom: 19 });
    map.setView([start.lat, start.lon], start.zoom);
    L.control.zoom({ zoomInTitle: data.zoomIn, zoomOutTitle: data.zoomOut }).addTo(map);
    const tiles = L.tileLayer("https://tile.openstreetmap.org/{z}/{x}/{y}.png", {
      maxZoom: 19,
      noWrap: true,
      attribution: '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors',
    }).addTo(map);
    if (resultsMap) {
      status.textContent = "";
      tiles.on("tileerror", () => { status.textContent = data.tilesError; });
      tiles.on("load", () => {
        if (!tiles.getContainer()?.querySelector(".leaflet-tile-loaded")) {
          status.textContent = data.tilesError;
        } else {
          status.textContent = "";
        }
      });
      requestAnimationFrame(() => {
        if (closed) return;
        map.invalidateSize();
        showFacilityMarkers(L, map, data.markers, data.origin, data.pinIcon);
      });
      return;
    }
    let tileError = false;
    let hasTiles = false;
    let submitted = false;
    const refresh = () => {
      const zoomed = map.getZoom() >= 11;
      button.disabled = !zoomed || !hasTiles || submitted;
      status.textContent = tileError ? data.tilesError : zoomed ? "" : data.zoomForPin;
    };
    tiles.on("tileerror", () => { tileError = true; refresh(); });
    tiles.on("tileload", () => { hasTiles = true; tileError = false; refresh(); });
    map.on("zoomend", refresh);
    button.onclick = () => {
      const center = map.getCenter().wrap();
      submitted = true;
      refresh();
      status.textContent = data.checking;
      setTriggerValue("confirmed", { lat: center.lat, lon: center.lng, zoom: map.getZoom() });
    };
    requestAnimationFrame(() => { if (!closed) map.invalidateSize(); });
    refresh();
  }).catch(() => {
    if (!closed) status.textContent = data.loadError;
  });

  return () => {
    closed = true;
    button.onclick = null;
    if (map) map.remove();
  };
}
~~~~

### map_picker.css

~~~~css
.picker-frame {
  position: relative;
  isolation: isolate;
}
.picker-map {
  height: clamp(240px, 54vh, 500px);
  border-radius: 0.5rem;
  background: #e5e9e4;
}
.picker-pin {
  position: absolute;
  left: 50%;
  top: 50%;
  width: 52px;
  height: 52px;
  transform: translate(-26px, -46px);
  z-index: 1000;
  pointer-events: none;
}

.picker-pin,
.search-origin-pin {
  filter: drop-shadow(0 1px 2px #0005);
}

.picker-actions {
  display: flex;
  gap: 0.75rem;
  align-items: center;
  justify-content: space-between;
  margin-top: 0.75rem;
}
.picker-confirm {
  border: 0;
  border-radius: 0.5rem;
  background: #ad2638;
  color: white;
  padding: 0.48rem 0.8rem;
  font: inherit;
  cursor: pointer;
  flex-shrink: 0;
}
.picker-confirm:disabled {
  opacity: 0.55;
  cursor: default;
}
.picker-status { font-size: 0.9rem; }

.leaflet-marker-icon.facility-map-marker {
  display: grid;
  place-items: center;
  border: 2px solid #a62336;
  border-radius: 50%;
  background: #fff;
  box-shadow: 0 1px 5px #0006;
  color: #a62336;
  font: 700 0.75rem/1 system-ui, sans-serif;
  white-space: nowrap;
}

.facility-location-warning {
  color: #854d0e;
  font-weight: 600;
}
~~~~

### ranking.css

~~~~css
.facility-rank-heading {
  display: flex;
  align-items: center;
  gap: 0.85rem;
  min-width: 0;
}

.facility-rank-heading h3 {
  margin: 0;
  padding: 0;
  font-size: 1.25rem;
  line-height: 1.32;
  overflow-wrap: anywhere;
}

.rank-badge {
  position: relative;
  isolation: isolate;
  display: grid;
  place-items: center;
  flex: 0 0 2.8rem;
  width: 2.8rem;
  height: 2.8rem;
  font-variant-numeric: tabular-nums;
}

.rank-badge__number {
  display: grid;
  place-items: center;
  width: 2.1rem;
  height: 2.1rem;
  border: 1px solid #c5ccd6;
  border-radius: 50%;
  background: #edf0f5;
  color: #364356;
  font-size: 0.9rem;
  font-weight: 750;
}

.rank-badge--gold {
  --rim: #a16612;
  --light: #fff1ba;
  --metal: #e1a43d;
  --ink: #593700;
  --ribbon: #a54624;
}

.rank-badge--silver {
  --rim: #78879a;
  --light: #f6faff;
  --metal: #a8b5c7;
  --ink: #344156;
  --ribbon: #4d6086;
}

.rank-badge--bronze {
  --rim: #915232;
  --light: #fce1cf;
  --metal: #c98256;
  --ink: #5f301b;
  --ribbon: #83563b;
}

.rank-badge:is(.rank-badge--gold, .rank-badge--silver, .rank-badge--bronze)::before {
  content: "";
  position: absolute;
  z-index: -1;
  inset: 1.75rem 0.2rem -0.28rem;
  background: var(--ribbon);
  clip-path: polygon(17% 0, 83% 0, 100% 100%, 72% 82%, 54% 100%, 50% 55%, 46% 100%, 28% 82%, 0 100%);
}

.rank-badge:is(.rank-badge--gold, .rank-badge--silver, .rank-badge--bronze) .rank-badge__number {
  position: relative;
  box-sizing: border-box;
  width: 2.56rem;
  height: 2.56rem;
  border: 1px solid var(--rim);
  background: radial-gradient(ellipse at 28% 16%, #fff9 0%, transparent 65%),
      linear-gradient(145deg, var(--light), var(--metal));
  box-shadow: inset 0 0 0 2px var(--light), inset 0 0 0 3px var(--rim),
      0 2px 5px #20232b40;
  color: var(--ink);
  font-size: 1.05rem;
  font-weight: 800;
}

.rank-evidence-warning {
  display: inline-grid;
  place-items: center;
  width: 1.05rem;
  height: 1.05rem;
  margin-inline-start: 0.25em;
  border-radius: 50%;
  background: #b42336;
  color: #fff;
  font-size: 0.72rem;
  font-weight: 800;
  line-height: 1;
  vertical-align: 0.12em;
}
~~~~

### app.yaml

~~~~yaml
command: ["streamlit", "run", "app.py"]
env:
  - name: DATABRICKS_WAREHOUSE_ID
    valueFrom: sql-warehouse
  - name: STREAMLIT_GATHER_USAGE_STATS
    value: "false"
~~~~

### requirements.txt

~~~~text
streamlit>=1.63,<2
databricks-sdk>=0.38,<1
databricks-sql-connector>=4,<5
~~~~